<a href="https://colab.research.google.com/github/Wasayyyyyy/CV-Tasks/blob/main/landmark_recognition_lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Landmark Recognition Using SIFT (OpenCV)

Pipeline: **database -> SIFT features -> query processing -> BF + kNN (k=2) + Lowe ratio test -> RANSAC homography localization -> evaluation**.

Install once: `pip install opencv-python numpy matplotlib`

Folder layout expected next to this notebook:
```
database/
    Eiffel_Tower/  Taj_Mahal/  Great_Wall/  Statue_of_Liberty/  Colosseum/   (several photos each)
```

## 0. Imports and configuration

In [ ]:
   !unzip -q data.zip

In [ ]:
import csv, pickle, time
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DB_DIR   = Path("database")
DB_FILE  = Path("sift_database.pkl")
OUT_DIR  = Path("results")
OUT_DIR.mkdir(exist_ok=True)

MAX_DIM          = 1000   # downscale so longest side <= MAX_DIM
RATIO            = 0.75   # Lowe's ratio test
RANSAC_THRESH    = 5.0    # pixels
# Decision thresholds (reject unknown / wrong matches)
MIN_GOOD_MATCHES = 25     # reliable matches needed
MIN_INLIERS      = 12     # RANSAC inliers needed
MIN_INLIER_RATIO = 0.25   # inliers / reliable matches
MARGIN           = 1.5    # winner inliers must beat runner-up by this factor

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 1. Helper functions

In [ ]:
def load_image(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return img

def limit_size(img, max_dim=MAX_DIM):
    h, w = img.shape[:2]
    s = max_dim / max(h, w)
    if s < 1.0:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return img

def show(img_bgr, title="", figsize=(10, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

def extract_sift(img_bgr):
    """Resize -> grayscale -> SIFT keypoints + 128-D descriptors."""
    img = limit_size(img_bgr)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    kp, desc = sift.detectAndCompute(gray, None)
    return img, list(kp), desc

def draw_keypoints(img, kp):
    return cv2.drawKeypoints(img, kp, None, color=(0, 255, 0),
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

def pack_kp(kps):      # cv2.KeyPoint is not picklable
    return [(k.pt, k.size, k.angle, k.response, k.octave, k.class_id) for k in kps]

def unpack_kp(data):
    return [cv2.KeyPoint(x=p[0], y=p[1], size=s, angle=a, response=r,
                         octave=o, class_id=c) for (p, s, a, r, o, c) in data]

## 2. Task 1 - Check the landmark database
Make sure there is one folder per landmark with several images each.

In [ ]:
assert DB_DIR.is_dir(), f"Create the folder '{DB_DIR}/' with one sub-folder per landmark."
for folder in sorted(p for p in DB_DIR.iterdir() if p.is_dir()):
    n = len([f for f in folder.iterdir() if f.suffix.lower() in IMG_EXT])
    print(f"{folder.name:<22} {n} images")

## 3. Task 2 - SIFT feature extraction for reference images
Load -> grayscale -> detect keypoints -> descriptors -> display -> store with landmark labels.

In [ ]:
SHOW_ALL_REFERENCES = False   # True = display every reference image; False = first image per landmark

database = {}
(OUT_DIR / "reference_keypoints").mkdir(exist_ok=True)

for folder in sorted(p for p in DB_DIR.iterdir() if p.is_dir()):
    refs = []
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() not in IMG_EXT:
            continue
        img, kp, desc = extract_sift(load_image(f))
        if desc is None or len(kp) < 2:
            print(f"[skip] {f} (no usable features)")
            continue
        refs.append({"file": str(f), "shape": img.shape[:2],
                     "kp": pack_kp(kp), "desc": desc})
        vis = draw_keypoints(img, kp)
        cv2.imwrite(str(OUT_DIR / "reference_keypoints" / f"{folder.name}_{f.stem}.jpg"), vis)
        if SHOW_ALL_REFERENCES or len(refs) == 1:
            show(vis, f"{folder.name}/{f.name} - {len(kp)} keypoints", (8, 5))
        print(f"{folder.name}/{f.name}: {len(kp)} keypoints, descriptors {desc.shape}")
    if refs:
        database[folder.name] = refs

with open(DB_FILE, "wb") as fh:
    pickle.dump(database, fh)
print(f"\nStored {sum(len(v) for v in database.values())} images from {len(database)} landmarks -> {DB_FILE}")

## 4. Task 4 - Matching functions
Brute-Force matcher (Euclidean/L2), kNN with k=2, Lowe's ratio test.

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2)

def ratio_matches(q_desc, r_desc):
    if q_desc is None or r_desc is None or len(q_desc) < 2 or len(r_desc) < 2:
        return []
    pairs = bf.knnMatch(q_desc, r_desc, k=2)
    return [p[0] for p in pairs if len(p) == 2 and p[0].distance < RATIO * p[1].distance]

## 5. Task 5 - Homography, RANSAC and localization

In [ ]:
def estimate_homography(ref_kp, q_kp, good):
    """Reference -> query homography with RANSAC. Returns (H, inlier_mask, n_inliers)."""
    if len(good) < 4:
        return None, None, 0
    src = np.float32([ref_kp[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([q_kp[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_THRESH)
    if H is None or mask is None:
        return None, None, 0
    return H, mask.ravel().astype(np.uint8), int(mask.sum())

def project_corners(H, ref_shape):
    h, w = ref_shape
    corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(corners, H)

def geometry_is_valid(H, ref_shape, q_shape):
    """Reject degenerate / implausible projections."""
    if H is None or not np.all(np.isfinite(H)):
        return False, None
    poly = project_corners(H, ref_shape)
    if not np.all(np.isfinite(poly)):
        return False, None
    pts = poly.reshape(-1, 2).astype(np.float32)
    if not cv2.isContourConvex(pts.reshape(-1, 1, 2)):
        return False, poly
    area, q_area = cv2.contourArea(pts), q_shape[0] * q_shape[1]
    if area < 0.01 * q_area or area > 6.0 * q_area:
        return False, poly
    return True, poly

## 6. Task 3 + 4 + 5 - Full recognition pipeline
Each landmark is scored by its **best reference image**. Candidates are ranked by reliable matches, but the **decision is not based on match count alone**: it uses RANSAC inliers, polygon validity, an inlier ratio, and a margin over the runner-up.

In [ ]:
def recognize(query_bgr, database):
    t0 = time.perf_counter()
    q_img, q_kp, q_desc = extract_sift(query_bgr)

    per = {}
    for name, refs in database.items():
        best = None
        for r in refs:
            good = ratio_matches(q_desc, r["desc"])
            if best is None or len(good) > len(best["good"]):
                best = {"ref": r, "good": good}
        ref_kp = unpack_kp(best["ref"]["kp"])
        H, mask, inl = estimate_homography(ref_kp, q_kp, best["good"])
        valid, poly = geometry_is_valid(H, best["ref"]["shape"], q_img.shape[:2])
        best.update(ref_kp=ref_kp, H=H, mask=mask, inliers=inl, valid=valid, poly=poly)
        per[name] = best

    ranking = sorted(per, key=lambda n: len(per[n]["good"]), reverse=True)
    verified = [n for n in per if per[n]["valid"] and per[n]["inliers"] >= MIN_INLIERS]
    cand = max(per, key=lambda n: per[n]["inliers"])
    b, n_good = per[cand], len(per[cand]["good"])

    predicted, reason = "Unknown", ""
    if n_good < MIN_GOOD_MATCHES:
        reason = f"only {n_good} reliable matches (< {MIN_GOOD_MATCHES})"
    elif cand not in verified:
        reason = f"geometric verification failed ({b['inliers']} inliers, need >= {MIN_INLIERS} and a valid polygon)"
    elif b["inliers"] / max(n_good, 1) < MIN_INLIER_RATIO:
        reason = f"low inlier ratio ({b['inliers']}/{n_good})"
    else:
        others = [per[n]["inliers"] for n in per if n != cand]
        second = max(others) if others else 0
        if second > 0 and b["inliers"] < MARGIN * second:
            reason = f"ambiguous: runner-up has {second} inliers vs {b['inliers']} for {cand}"
        else:
            predicted = cand

    return {"query_img": q_img, "q_kp": q_kp, "n_kp": len(q_kp), "per": per,
            "ranking": ranking, "best": cand, "predicted": predicted,
            "recognized": predicted != "Unknown", "reason": reason,
            "time": time.perf_counter() - t0, "n_good": n_good, "n_inliers": b["inliers"]}

def draw_result(res, title_prefix=""):
    """Matches + localized polygon (or a rejection message)."""
    name = res["best"]; b = res["per"][name]
    q = res["query_img"].copy()
    ref_img = limit_size(load_image(b["ref"]["file"]))

    if res["recognized"]:
        cv2.polylines(q, [np.int32(b["poly"])], True, (0, 0, 255), 4, cv2.LINE_AA)
        mask = b["mask"].tolist()
        label, color = f"RECOGNIZED: {res['predicted']}", (0, 160, 0)
    else:
        mask = None
        label, color = f"REJECTED (closest: {name})", (0, 0, 200)

    vis = cv2.drawMatches(ref_img, b["ref_kp"], q, res["q_kp"],
                          [cv2.DMatch(m.trainIdx, m.queryIdx, m.distance) for m in b["good"]],
                          None, matchColor=(0, 255, 0), matchesMask=mask,
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    info = [f"{title_prefix}{label}",
            f"Reliable matches: {res['n_good']}   RANSAC inliers: {res['n_inliers']}",
            f"Time: {res['time']:.2f}s"]
    if not res["recognized"]:
        info.append(f"Reason: {res['reason']}")
        if not b["valid"]:
            info.append("Geometric transformation unreliable - no polygon drawn.")
    pad = 28 * len(info) + 12
    canvas = np.full((vis.shape[0] + pad, vis.shape[1], 3), 255, np.uint8)
    canvas[pad:] = vis
    for i, t in enumerate(info):
        cv2.putText(canvas, t, (10, 24 + 28 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                    color if i == 0 else (30, 30, 30), 2 if i == 0 else 1, cv2.LINE_AA)
    return canvas

def print_report(res):
    print("Candidate ranking (reliable matches / RANSAC inliers):")
    for i, n in enumerate(res["ranking"], 1):
        p = res["per"][n]
        print(f"  {i}. {n:<22} matches={len(p['good']):>4}  inliers={p['inliers']:>4}")
    print(f"Query keypoints : {res['n_kp']}")
    print(f"Reliable matches: {res['n_good']} | RANSAC inliers: {res['n_inliers']}")
    if res["recognized"]:
        print(f"RESULT: landmark RECOGNIZED -> {res['predicted']}")
    else:
        print(f"RESULT: REJECTED / unknown landmark ({res['reason']})")
    print(f"Processing time : {res['time']:.3f} s")

## 7. Recognize a single query image (Tasks 3, 4, 5)
Set `QUERY_PATH` to your unknown landmark photo.

In [ ]:
QUERY_PATH = "queries/query.jpg"

query = load_image(QUERY_PATH)
show(limit_size(query), "1. Query image", (8, 5))

res = recognize(query, database)
show(draw_keypoints(res["query_img"], res["q_kp"]),
     f"2. Query SIFT keypoints ({res['n_kp']})", (8, 5))

result_img = draw_result(res)
show(result_img, "3. Matches, predicted landmark and localized region", (14, 7))
print_report(res)
cv2.imwrite(str(OUT_DIR / "query_result.jpg"), result_img)

## 8. Task 6 - Performance evaluation
Six test conditions. Rotated / resized / darker versions are generated from `ORIGINAL`.
`ORIGINAL` should be an image **of a landmark in the database**, `VIEWPOINT` the same landmark from another viewpoint, and `UNKNOWN` a landmark that is **not** in the database.

In [ ]:
ORIGINAL  = "queries/original.jpg"
VIEWPOINT = "queries/viewpoint.jpg"
UNKNOWN   = "queries/unknown.jpg"

def rotate_bound(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(img, M, (nw, nh), borderValue=(0, 0, 0))

base = load_image(ORIGINAL)
tests = [
    ("Original", base),
    ("Rotated 45 deg", rotate_bound(base, 45)),
    ("Resized 50%", cv2.resize(base, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)),
    ("Reduced brightness (x0.4)", cv2.convertScaleAbs(base, alpha=0.4, beta=0)),
    ("Different viewpoint", load_image(VIEWPOINT)),
    ("Landmark not in database", load_image(UNKNOWN)),
]

rows = []
for i, (name, img) in enumerate(tests, 1):
    res = recognize(img, database)
    out = draw_result(res, title_prefix=f"[{name}] ")
    cv2.imwrite(str(OUT_DIR / f"test{i}_result.jpg"), out)
    show(out, f"Test {i}: {name}", (14, 7))
    print_report(res)
    print("-" * 60)
    rows.append({
        "Test": name,
        "Predicted landmark": res["predicted"] if res["recognized"] else f"Unknown (closest: {res['best']})",
        "Query keypoints": res["n_kp"],
        "Reliable matches": res["n_good"],
        "RANSAC inliers": res["n_inliers"],
        "Decision": "Recognized" if res["recognized"] else "Rejected",
        "Time (s)": f"{res['time']:.2f}",
    })

### Results table

In [ ]:
hdr = list(rows[0].keys())
table = ["| " + " | ".join(hdr) + " |", "|" + "---|" * len(hdr)]
table += ["| " + " | ".join(str(r[h]) for h in hdr) + " |" for r in rows]
from IPython.display import Markdown, display
display(Markdown("\n".join(table)))

with open(OUT_DIR / "evaluation.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=hdr); w.writeheader(); w.writerows(rows)